# ⛑️ Safety Helmet Detection — 2-Class YOLO Training (helmet & no-helmet)

Train a high-accuracy custom YOLOv8/YOLOv11 model for **2 Classes**:
- **Class 0 (`helmet`)**: Person wearing safety helmet / hard-hat (Compliant)
- **Class 1 (`no-helmet`)**: Person without helmet / bare head (Safety Violation)

> **Important**: Enable GPU in Google Colab before running:
> `Runtime` ➔ `Change runtime type` ➔ select **T4 GPU** (or A100/V100 if available).

In [ ]:
# Step 1: Check GPU & Install Required Libraries
!nvidia-smi
!pip install -q ultralytics roboflow pyyaml

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")

## 📥 Step 2: Download Dataset
Choose **Option A (Roboflow Export Code)** OR **Option B (Upload Dataset Zip)**.

In [ ]:
# =========================================================================
# OPTION A: Roboflow Direct Download
# =========================================================================
from roboflow import Roboflow

# Replace with your API key & project details if using Roboflow:
ROBOFLOW_API_KEY = "AydYTkTEwRNm3fM0n8yl"  # <-- Replace with your API key
WORKSPACE_NAME = "YOUR_WORKSPACE"        # <-- Replace with workspace name
PROJECT_NAME = "safety-helmet-dataset"   # <-- Replace with project name
VERSION = 1

try:
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(WORKSPACE_NAME).project(PROJECT_NAME)
    version = project.version(VERSION)
    dataset = version.download("yolov8")
    data_yaml_path = f"{dataset.location}/data.yaml"
    print(f"Dataset downloaded successfully to: {dataset.location}")
except Exception as e:
    print(f"Option A not configured or failed ({e}). Use Option B if uploading zip manually.")

In [ ]:
# =========================================================================
# OPTION B: Upload Dataset Zip File (If you downloaded zip from Roboflow or local)
# =========================================================================
import os, glob, zipfile
from google.colab import files

# Only run if Option A was not used or failed
if 'data_yaml_path' not in locals() or not os.path.exists(data_yaml_path):
    print("Upload your dataset .zip file from your computer:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            extract_dir = '/content/dataset'
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall(extract_dir)
            print(f"Extracted {filename} to {extract_dir}")

    found_yamls = glob.glob('/content/dataset/**/data.yaml', recursive=True) + glob.glob('/content/**/data.yaml', recursive=True)
    if found_yamls:
        data_yaml_path = found_yamls[0]
        print(f"Found data.yaml at: {data_yaml_path}")

## 🔄 Step 3: Standardize & Convert Dataset to Exactly 2 Classes (`helmet` & `no-helmet`)

Many safety helmet datasets contain extra classes (e.g. `person`, `vest`) or different naming schemes (`head`, `hard-hat`, `without_helmet`).

This cell automatically:
1. Remaps all helmet variations (`helmet`, `hardhat`, `with_helmet`, etc.) to **`0: helmet`**.
2. Remaps all bare head / violation variations (`head`, `no-helmet`, `without_helmet`, `bare_head`, etc.) to **`1: no-helmet`**.
3. Safely ignores non-head classes (e.g. `person`) so the model specializes purely on head protection.
4. Updates all label txt files and configures `data.yaml` with `nc: 2` and `names: ["helmet", "no-helmet"]`.

In [ ]:
import os, glob, yaml
from collections import Counter

with open(data_yaml_path, 'r') as f:
    data_cfg = yaml.safe_load(f)

raw_names = data_cfg.get('names')
if isinstance(raw_names, dict):
    orig_id_to_name = {int(k): str(v).lower().strip() for k, v in raw_names.items()}
elif isinstance(raw_names, list):
    orig_id_to_name = {i: str(v).lower().strip() for i, v in enumerate(raw_names)}
else:
    raise ValueError(f"Unexpected names format in data.yaml: {raw_names}")

print("Original Dataset Classes:", orig_id_to_name)

# Synonym sets for exact 2-class alignment
HELMET_SYNONYMS = {"helmet", "hardhat", "hard hat", "hard-hat", "with_helmet", "with helmet", "safety helmet", "safety_helmet"}
NO_HELMET_SYNONYMS = {"head", "bare head", "bare_head", "no-helmet", "no_helmet", "without_helmet", "without helmet", "no helmet", "no-hardhat", "no_hardhat", "no hardhat", "cap", "hat"}

class_id_map = {}
for orig_id, name in orig_id_to_name.items():
    if name in HELMET_SYNONYMS:
        class_id_map[orig_id] = 0  # helmet
    elif name in NO_HELMET_SYNONYMS:
        class_id_map[orig_id] = 1  # no-helmet
    else:
        # Discard other classes (e.g. "person", "vest") to train directly on helmet vs no-helmet
        print(f"  Discarding non-head class '{name}' (id {orig_id}) to maintain 2-class focus.")
        class_id_map[orig_id] = None

print("Class Remapping Plan (0: helmet, 1: no-helmet):", class_id_map)

# Find all label txt files
dataset_root = os.path.dirname(os.path.abspath(data_yaml_path))
all_txts = glob.glob(f"{dataset_root}/**/*.txt", recursive=True)
label_files = [f for f in all_txts if not f.endswith('train.txt') and not f.endswith('val.txt') and not f.endswith('test.txt') and not f.endswith('README.txt') and not f.endswith('requirements.txt')]

class_counts = Counter()
modified_file_count = 0

for lf in label_files:
    if not os.path.isfile(lf):
        continue
    with open(lf, 'r') as f:
        lines = f.readlines()
    
    new_lines = []
    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue
        try:
            orig_cls = int(parts[0])
        except ValueError:
            continue
        if orig_cls in class_id_map and class_id_map[orig_cls] is not None:
            target_cls = class_id_map[orig_cls]
            new_lines.append(f"{target_cls} " + " ".join(parts[1:]) + "\n")
            class_counts[target_cls] += 1
    
    with open(lf, 'w') as f:
        f.writelines(new_lines)
    modified_file_count += 1

# Rewrite data.yaml to enforce 2 classes
data_cfg['nc'] = 2
data_cfg['names'] = ['helmet', 'no-helmet']
with open(data_yaml_path, 'w') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print("=" * 50)
print("✅ 2-CLASS DATASET READY:")
print(f"  Class 0 [helmet]    : {class_counts[0]} instances")
print(f"  Class 1 [no-helmet] : {class_counts[1]} instances")
print(f"  Processed {modified_file_count} label files")
print(f"  Updated configuration at: {data_yaml_path}")
print("=" * 50)

## 🚀 Step 4: Train 2-Class YOLO Model

We recommend `yolov8m.pt` (or `yolo11m.pt`) for high precision on small/distant helmets and heads.
Use `yolov8s.pt` if edge inference speed is prioritized.

In [ ]:
from ultralytics import YOLO

# Model selection: 'yolov8m.pt' (Medium) for high accuracy, or 'yolov8s.pt' for faster training
base_model = 'yolov8m.pt'
model = YOLO(base_model)

# Train with hyperparameters optimized for head & helmet detection
results = model.train(
    data=data_yaml_path,
    epochs=100,           # 80-120 epochs recommended
    imgsz=640,            # 640 or 1024 for small distant heads
    batch=16,
    name='helmet_detector_2class',
    patience=20,          # Early stopping
    augment=True,
    mosaic=1.0,           # Augmentation for multi-scale objects
    mixup=0.10,
    fliplr=0.5,           # Left-right flip
    flipud=0.0,
    degrees=10.0,         # Head tilt variations
    scale=0.5,            # Multi-scale distance variations
    hsv_h=0.015,          # Color jitter for various helmet colors (yellow, white, red, blue)
    hsv_s=0.7,
    hsv_v=0.4,
    save=True,
    plots=True,
    device=0 if torch.cuda.is_available() else 'cpu'
)

## 📊 Step 5: Validate Model & Check 2-Class Metrics

Inspect overall and class-specific metrics (`helmet` vs `no-helmet`).

In [ ]:
# Evaluate validation set performance
metrics = model.val()

print("=" * 60)
print("OVERALL 2-CLASS MODEL PERFORMANCE:")
print(f"  Overall mAP @ 0.50:       {metrics.box.map50:.4f} ({metrics.box.map50*100:.1f}%)")
print(f"  Overall mAP @ 0.50-0.95:  {metrics.box.map:.4f} ({metrics.box.map*100:.1f}%)")
print(f"  Overall Mean Precision:   {metrics.box.mp:.4f} ({metrics.box.mp*100:.1f}%)")
print(f"  Overall Mean Recall:      {metrics.box.mr:.4f} ({metrics.box.mr*100:.1f}%)")
print("=" * 60)

# Per-class performance breakdown
class_labels = ["helmet", "no-helmet"]
print("\nPER-CLASS ACCURACY BREAKDOWN:")
for i, label in enumerate(class_labels):
    if hasattr(metrics.box, 'maps') and i < len(metrics.box.maps):
        c_map50 = metrics.box.maps[i]
        c_p = metrics.box.p[i] if i < len(metrics.box.p) else 0.0
        c_r = metrics.box.r[i] if i < len(metrics.box.r) else 0.0
        print(f"  Class [{i}] {label:<12} -> Precision: {c_p*100:5.1f}% | Recall: {c_r*100:5.1f}% | mAP@50: {c_map50*100:5.1f}%")
print("=" * 60)

In [ ]:
# Display results curve & confusion matrix
from IPython.display import Image, display
import glob

plot_files = glob.glob('runs/detect/helmet_detector_2class*/results.png') + \
             glob.glob('runs/detect/helmet_detector_2class*/confusion_matrix.png') + \
             glob.glob('runs/detect/helmet_detector_2class*/val_batch0_pred.jpg')

for p in plot_files:
    print(f"Plot: {p}")
    display(Image(p))

## 👁️ Step 6: Visual Inference Demo on Test Images

Visualizes detections with color-coded bounding boxes:
- 🟢 **Green**: `helmet` (Safe / Compliant)
- 🔴 **Red**: `no-helmet` (Violation)

In [ ]:
import cv2
import glob
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Find sample images from test/val splits
dataset_dir = os.path.dirname(os.path.abspath(data_yaml_path))
candidate_images = glob.glob(f"{dataset_dir}/test/images/*.jpg") + \
                   glob.glob(f"{dataset_dir}/valid/images/*.jpg") + \
                   glob.glob(f"{dataset_dir}/val/images/*.jpg")

if candidate_images:
    samples = candidate_images[:3]
    print(f"Visualizing predictions on {len(samples)} sample images:")
    
    for img_path in samples:
        preds = model.predict(img_path, conf=0.35, imgsz=640)[0]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        helmets_found = 0
        violations_found = 0
        
        for b in preds.boxes:
            cls_id = int(b.cls[0])
            conf = float(b.conf[0])
            x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
            
            if cls_id == 0:
                color = (0, 255, 0)      # Green for helmet
                label = f"Helmet {conf:.2f}"
                helmets_found += 1
            else:
                color = (255, 0, 0)      # Red for no-helmet violation
                label = f"NO-HELMET {conf:.2f}"
                violations_found += 1
            
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, label, (x1, max(y1 - 8, 15)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
        
        plt.figure(figsize=(9, 6))
        plt.title(f"{os.path.basename(img_path)} | Helmets: {helmets_found} | Violations: {violations_found}")
        plt.imshow(img)
        plt.axis('off')
        plt.show()
else:
    print("No test/val images found in standard folders. Inference cell ready for custom image paths.")

## 💾 Step 7: Download `best.pt` Weights File

Download `best.pt` and place it in your local service folder:
`apps/helmet_detection/best.pt`

In [ ]:
import glob
from google.colab import files

best_weights = glob.glob('runs/detect/helmet_detector_2class*/weights/best.pt')
if best_weights:
    latest_best = best_weights[-1]
    print(f"Downloading {latest_best} to your local computer...")
    files.download(latest_best)
else:
    print("Error: best.pt weights not found. Please verify runs/detect directory.")